# Online Design Experiment Tutorial

This tutorial demonstrates how to run an online design experiment using the ALF (Active Learning Framework) codebase. We'll walk through the complete process of setting up and running an online active learning experiment for protein design, using a fast *in-silico* oracle to score newly generated sequences.

### What is an Online Design Experiment?

An online design experiment is a type of active learning experiment where we query a real oracle (such as a physics simulator or wet-lab experiment) to obtain ground truth labels in real time. Unlike offline experiments that use a fixed dataset, online experiments:
- Generate new labels on-the-fly as sequences are proposed
- Can explore beyond pre-existing labelled data
- More closely simulate real-world experiments
- Are typically more computationally expensive per query

### Experiment Overview

In this tutorial, we'll touch on each part of an online design experiment:
- Set up the GFP dataset to provide the initial labelled data
- Stand in a fast in-silico oracle (a model trained on the full dataset) for the expensive real-time evaluation
- Initialise a CNN surrogate model to approximate the oracle and guide the search
- Generate new candidate sequences with a single-mutant search strategy
- Run multiple rounds of active learning with live oracle queries
- Analyse how efficiently we discover high-fitness proteins

> **A note on the oracle.** A *real* online oracle is expensive — a physics engine such as PyRosetta, or a wet-lab assay. To keep this tutorial fast and installable with a plain `uv sync` (no multi-gigabyte, ~30-minute builds), we replace it with a cheap in-silico oracle: a model trained on the *full* GFP dataset that stands in for ground truth. The active learning machinery is identical — only the scorer behind the `Oracle` changes.

### Framework Components

Before we start, let's understand the key components of the ALF framework:

1. **Dataset** ([`GFP`](https://instadeepai.github.io/alf/api/alf_tools/datasets/gfp/)): Contains the initial labelled data for training. Handles data splitting into train/validation/test sets. In online experiments, the candidate pool is empty since candidates are generated rather than pre-defined.

2. **Surrogate Model** ([`CNNModel`](https://instadeepai.github.io/alf/api/alf_tools/models/cnn/)): Trained on labelled data to approximate the expensive oracle evaluation and guide candidate selection.

3. **Search Strategy** ([`SingleMutantSearch`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/search/single_mutant_search/)): Generates new candidate sequences by systematically mutating the current best sequence. Unlike offline experiments, candidates are created on-the-fly rather than drawn from a fixed pool.

4. **Acquisition Function** ([`Greedy`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/greedy/)): Determines which generated sequences are most promising to evaluate next based on surrogate model predictions.

5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): Combines acquisition function and search strategy and handles the "ask" (generate and select candidates) and "tell" (update surrogate with results) cycle.

6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/)): The evaluation function that returns a fitness value for each proposed sequence. In a real online experiment this is an expensive query — a physics simulator or a wet-lab assay. Here we use a fast in-silico stand-in: a [`CNNModel`](https://instadeepai.github.io/alf/api/alf_tools/models/cnn/) trained on the full GFP dataset, so the loop runs quickly with no heavy dependencies. The surrogate never sees the oracle's weights — only the labels it returns.

7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): Orchestrates the entire active learning loop, managing multiple acquisition rounds, oracle queries, and model updates.

### Step 0: Setup

These tutorials are written for **dev mode** — running from a local clone of the ALF repository. From the `tutorials/` directory:

```bash
uv sync   # installs alf_core, alf_tools and tutorial deps (CPU PyTorch by default)
```

Register the environment as a Jupyter kernel, then select the `alf` kernel in this notebook:

```bash
uv run ipython kernel install --user --env VIRTUAL_ENV "$(pwd)/.venv" --name=alf
```

For GPU support and full details, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

**Not running from a clone?** If you installed ALF with `pip`, run the optional cell below to install this tutorial's dependencies into the current kernel.

In [ ]:
# Optional — only needed if you are NOT running from a cloned repo via `uv sync`.
# Installs ALF and this tutorial's dependencies into the current kernel, then restart the kernel.
# %pip install alf_core alf_tools matplotlib pandas

### Step 1: Import Required Libraries

Let's start by importing all the necessary components from the ALF framework:


In [ ]:
# Core framework imports
import shutil

import matplotlib.pyplot as plt

# Additional imports for analysis
import pandas as pd
from alf_core import (
    DesignTask,
    FileStateLogger,
    Optimizer,
    Oracle,
    ProtocolSearch,
    Surrogate,
    TerminalStateLogger,
)
from alf_tools.datasets import GFP
from alf_tools.models import CNNModel
from alf_tools.optimizer.acquisition_functions import Greedy
from alf_tools.optimizer.search import SingleMutantSearch

print("✅ All imports successful!")

### Step 2: Configure the Dataset

The [`GFP`](https://instadeepai.github.io/alf/api/alf_tools/datasets/gfp/) dataset contains protein sequences and their measured brightness values. Let's set up the dataset with appropriate splits using [`BaseDatasetConfig`](https://instadeepai.github.io/alf/api/alf_core/dataset/base_dataset/) and [`Modality`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/):

In [ ]:
# Dataset configuration
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDatasetConfig
from alf_core.utils.enums import ProblemType

# Initialize the GFP dataset
gfp_dataset = GFP(
    BaseDatasetConfig(
        name="gfp",
        modality=Modality.SEQUENCE,  # We're working with protein sequences
        seed=51505,  # For reproducibility
        train_ratio=0.1,
        validation_frac=0.2,
        test_ratio=0.1,
        split_type="random",
        problem_type=ProblemType.REGRESSION,  # GFP brightness is a continuous score
    )
)

print("✅ Dataset initialized!")
print("📊 Dataset info:")
print(f"   - Total sequences: {len(gfp_dataset._raw_dataset)}")
print(f"   - Sequence length: {len(gfp_dataset._raw_dataset.candidates[0].data)}")
print(
    f"   - Fitness score range: {gfp_dataset._raw_dataset.labels.min():.2f} to"
    f" {gfp_dataset._raw_dataset.labels.max():.2f}"
)

### Step 3: Initialize the Surrogate Model

The surrogate model is a CNN that learns to predict protein fitness from sequence. We wrap it in a [`Surrogate`](https://instadeepai.github.io/alf/api/alf_core/surrogate/) - this is the "brain" of our active learning system:

In [ ]:
# Initialize the CNN surrogate model
surrogate = Surrogate(model=CNNModel())

print("✅ Surrogate model initialized!")
print("🧠 Model architecture:")
print("   - Type: 1D Convolutional Neural Network")
print("   - Input: One-hot encoded protein sequences")
print("   - Output: Predicted fitness values")
print("   - Purpose: Learn sequence → fitness mapping")

### Step 4: Set Up the Acquisition Strategy

The acquisition function determines which sequences are most promising to evaluate next. We'll use a simple greedy strategy:


In [ ]:
# Initialize acquisition function (greedy strategy)
acquisition_fn = Greedy()

print("✅ Acquisition function initialized!")
print("🎯 Strategy: Greedy selection")
print("   - Selects sequences with highest predicted fitness")
print("   - Simple but effective for many design tasks")
print("   - Alternative strategies: UCB, Thompson sampling, etc.")

### Step 5: Configure the Search Strategy

The search strategy defines how we generate new sequences. We'll use [`ProtocolSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/) with a [`SingleMutantSearch`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/search/single_mutant_search/) protocol, which proposes every single-point mutant of the current best sequence.

The GFP sequences in this dataset are nucleotide sequences, so we mutate over the DNA alphabet (`"ACGT"`) — this keeps the generated candidates in the same alphabet the surrogate and oracle were trained on. (`SingleMutantSearch` defaults to the 20-letter protein alphabet, which is the right choice when your candidates are amino-acid sequences.)

In [ ]:
# Initialize search strategy (mutate over the DNA alphabet, matching the GFP sequences)
search_fn = ProtocolSearch(protocol=SingleMutantSearch(alphabet="ACGT"))

print("✅ Search strategy initialized!")
print("🔍 Strategy: Single mutant search")
print("   - Generates all single-point mutations of the best sequence")
print("   - Explores the local neighbourhood around the current best")
print("   - Mutates over the nucleotide alphabet (A/C/G/T)")

### Step 6: Create the Optimizer

The optimizer combines the acquisition function and search strategy to handle the ask-tell cycle:


In [ ]:
# Initialize optimizer
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

print("✅ Optimizer initialized!")
print("⚙️ Optimizer components:")
print("   - Acquisition function: Greedy selection")
print("   - Search strategy: Dataset-based search")
print("   - Handles: ask() → select candidates, tell() → update surrogate with results")

### Step 7: Set Up the Oracle

The [`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/) provides the ground-truth fitness for each proposed sequence. In a real online experiment this is the expensive part — a physics simulator or a wet-lab assay.

To keep this tutorial fast and free of heavy dependencies, we stand in a cheap **in-silico oracle**: a [`CNNModel`](https://instadeepai.github.io/alf/api/alf_tools/models/cnn/) trained on the *full* GFP dataset. It plays the role of ground truth — the surrogate is trained only on the small labelled splits and the candidates acquired during the loop, and it learns to approximate this oracle. Any [`BaseModel`](https://instadeepai.github.io/alf/api/alf_core/model/base_model/) (or a [`BaseDataset`](https://instadeepai.github.io/alf/api/alf_core/dataset/base_dataset/), as in the offline tutorial) can be used as the oracle's scorer.

We give the oracle more training epochs than the per-round surrogate so that it is a reasonably accurate stand-in for ground truth.

In [ ]:
from alf_core import LabelledCandidates
from alf_tools.models import CNNTrainConfig

# Train the in-silico oracle on the entire labelled GFP dataset.
full_data = gfp_dataset.load_dataset()
n_val = max(1, len(full_data.candidates) // 10)
oracle_train = LabelledCandidates(
    candidates=full_data.candidates[:-n_val], labels=full_data.labels[:-n_val]
)
oracle_val = LabelledCandidates(
    candidates=full_data.candidates[-n_val:], labels=full_data.labels[-n_val:]
)

# A few more epochs than the per-round surrogate, so the oracle is a decent stand-in
# for ground truth. This trains once, up front; the surrogate then learns to chase it.
oracle_model = CNNModel(train_config=CNNTrainConfig(num_epochs=20))
oracle_model.setup(gfp_dataset)
print("🔬 Training in-silico oracle on the full GFP dataset...")
oracle_model.train(oracle_train, oracle_val)

oracle = Oracle(scorer=oracle_model)

print("✅ Oracle ready!")
summary = oracle_model.get_training_summary_metrics()
print(f"   - Oracle val Spearman: {summary['final_val_spearman']:.3f}")
print("   - Returns a fitness for any proposed sequence (stands in for the wet-lab/physics oracle)")

### Step 8: Configure the Design Task

Now we'll set up the main design task that orchestrates the entire active learning process:


In [ ]:
# Configure the design task
num_acq_rounds = 4  # Number of active learning rounds
acq_batch_size = 2  # Number of sequences to acquire per round

task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)

print("✅ Design task configured!")
print("📋 Experiment parameters:")
print(f"   - Acquisition rounds: {num_acq_rounds}")
print(f"   - Batch size per round: {acq_batch_size}")
print(f"   - Total sequences to acquire: {num_acq_rounds * acq_batch_size}")
print("   - Task type: Online design optimization")

### Step 9: Run the Online Design Experiment

Now let's run the complete experiment! This will:
1. Set up the initial state with training data
2. Run multiple rounds of active learning
3. Track performance metrics throughout


In [ ]:
import logging
from pathlib import Path

# Keep the output readable: only surface warnings and above from the framework.
# Raise this to logging.INFO if you want the full per-round trace.
logging.basicConfig(level=logging.WARNING)

# Initialize logger for tracking metrics
terminal_logger = TerminalStateLogger()
save_path = Path("results/online_design/")
if save_path.exists():
    shutil.rmtree(save_path)
file_logger = FileStateLogger(output_path=save_path)
loggers = [terminal_logger, file_logger]

# Set up the initial state and run the experiment
print("🚀 Setting up experiment...")
state = task.setup(dataset=gfp_dataset, surrogate=surrogate)
print(
    f"   train={len(state.dataset.train_dataset)}, "
    f"val={len(state.dataset.validation_dataset)}, "
    f"test={len(state.dataset.test_dataset)}, "
    f"pool={len(state.dataset.candidate_pool)}"
)

print("🔄 Running online active learning experiment...")
task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)
print("✅ Experiment completed!")

### Step 10: Analyze the Results

After the experiment completes, we can load the saved metrics and visualize the optimization progress. We track:
- **Round Mean Fitness**: Average fitness of selected sequences in each acquisition batch
- **Round Maximum Fitness**: Highest fitness value among sequences selected in each round

If the optimization is working well, these metrics should increase over rounds as we discover better sequences.

In [ ]:
# Load metrics from CSV
metrics = pd.read_csv("results/online_design/metrics.csv")

# Row 0 is the zeroth (initial) round with no acquisitions, and the final row is a trailing
# experiment-summary row that only fills aggregate columns (NaN for the per-round metrics).
# So the first acquisition round is iloc[1] and the last is iloc[-2].
print("Experiment Summary:")
print(f"Total Rounds (including zeroth round): {len(metrics)}")
print(f"Initial Mean Fitness: {metrics['acquired_candidates/round_mean'].iloc[1]:.4f}")
print(f"Final Mean Fitness: {metrics['acquired_candidates/round_mean'].iloc[-2]:.4f}")
print(f"Best Fitness Found: {metrics['acquired_candidates/round_max'].max():.4f}")

In [ ]:
# Create visualization of results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Online Protein Design Optimization Results", fontsize=16, fontweight="bold")

# Drop the zeroth round (first row) and the trailing experiment-summary row (last row).
# The DataFrame index is the round number.
metrics_plot = metrics.iloc[1:-1]

# 1. Round Mean Fitness (quality of acquired batch each round)
axes[0].plot(
    metrics_plot.index,
    metrics_plot["acquired_candidates/round_mean"],
    marker="o",
    linewidth=2,
    markersize=8,
    color="#e74c3c",
)
axes[0].set_xlabel("Round", fontsize=12)
axes[0].set_ylabel("Mean Fitness", fontsize=12)
axes[0].set_title("Acquired Batch Mean Fitness", fontsize=13, fontweight="bold")
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(metrics_plot.index)

# 2. Round Maximum Fitness (best fitness in acquired batch each round)
axes[1].plot(
    metrics_plot.index,
    metrics_plot["acquired_candidates/round_max"],
    marker="s",
    linewidth=2,
    markersize=8,
    color="#2ecc71",
)
axes[1].set_xlabel("Round", fontsize=12)
axes[1].set_ylabel("Maximum Fitness", fontsize=12)
axes[1].set_title("Acquired Batch Maximum Fitness", fontsize=13, fontweight="bold")
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(metrics_plot.index)

plt.tight_layout()
plt.show()

#### Interpreting the results

Both panels show the fitness of the batch acquired at each round, as scored by the oracle:

- **Mean Fitness** is the average over the batch — it tells us whether the loop is consistently proposing good sequences.
- **Maximum Fitness** is the single best sequence found that round — the quantity we ultimately care about in a design campaign.

Because each round mutates the current best and the greedy acquisition keeps pushing toward what the surrogate rates highest, we expect the maximum to climb (or at least not regress) over rounds. With only a handful of rounds, a tiny batch size, and a single-mutant neighbourhood, the curves are short and can be noisy — focus on the overall trend rather than any single point. Longer runs, larger batches, or a more exploratory acquisition function (e.g. `UCB`) typically find higher-fitness sequences.

In [ ]:
# Optionally, clean up the results directory
if save_path.exists():
    shutil.rmtree(save_path)

print("✅ Results directory cleaned up!")

## Conclusion

This tutorial has demonstrated how to run an **online design experiment** using the ALF framework. We've learned:

- **What online design experiments are** and how they differ from offline experiments
- **How to set up an oracle** for real-time sequence evaluation — here a fast in-silico stand-in (a model trained on the full dataset) in place of an expensive physics simulator or wet-lab assay
- **How to use generative search strategies** (`SingleMutantSearch`) that propose novel candidates beyond a fixed pool
- **How to run a complete active learning loop** with live oracle queries at each round
- **How to analyse the results** to evaluate experimental efficiency and optimization progress

The key distinction from offline experiments is that online design queries an oracle to label *newly generated* candidates in real time, rather than looking up pre-computed values from a fixed dataset. In a real campaign that oracle would be the expensive step — a physics engine such as PyRosetta, or a wet-lab experiment — and the whole point of the surrogate is to make as few of those queries as possible. Here we swapped in a cheap learned oracle so the loop runs in seconds and installs with a plain `uv sync`, but the machinery is identical: to plug in a real oracle, just pass a different scorer to `Oracle`.

The ALF framework provides a flexible and modular approach to active learning that can be adapted to many different design problems. Whether you're working on protein design, drug discovery, materials science, or other optimization tasks, the principles demonstrated here can be applied to guide efficient experimental design.

**Happy designing!** 🧬🔬✨